# 02 - DistilBERT training on Colab

Select **Runtime > Change runtime type > GPU**. Upload the private archive created by `python -m src.package_colab`. This notebook uses the exact baseline splits. Full training has not been run by preparing this notebook.

In [ ]:
import torch
assert torch.cuda.is_available(), "Select a GPU runtime before continuing"
print(torch.__version__, torch.cuda.get_device_name(0))

In [ ]:
from google.colab import files
from pathlib import Path
from zipfile import ZipFile
import os

workspace = Path("/content/customer-support-routing")
assert not workspace.exists(), "Use a fresh runtime or a new workspace directory"
uploaded = files.upload()
assert "distilbert-colab.zip" in uploaded, "Upload artifacts/distilbert-colab.zip"
workspace.mkdir()
with ZipFile("distilbert-colab.zip") as archive:
    for member in archive.infolist():
        target = (workspace / member.filename).resolve()
        assert target.is_relative_to(workspace.resolve()), "Unsafe archive path"
    archive.extractall(workspace)
os.chdir(workspace)
Path("data/predictions").mkdir(parents=True, exist_ok=True)

In [ ]:
%pip install -q -r requirements.txt transformers==4.57.6 accelerate==1.15.0

## Train using only training and validation data

The best checkpoint is selected by validation macro F1. Training and checkpoint download may take some time. Colab GPU availability varies.

In [ ]:
import subprocess
import sys
subprocess.run([sys.executable, "-m", "src.train_distilbert"], check=True)

In [ ]:
import json
metadata = json.loads(Path("reports/distilbert/training.json").read_text())
print(json.dumps({k: metadata[k] for k in ["status", "elapsed_seconds", "validation", "reload_inference_passed"]}, indent=2))

## Final held-out evaluation

Run after the full training finishes and checkpoint selection is fixed. Do not adjust parameters based on these test results. The command verifies split provenance against both the training run and the baseline artifact.

In [ ]:
subprocess.run([sys.executable, "-m", "src.train_distilbert", "--evaluate-only"], check=True)

## Download model and reports

Download before disconnecting the runtime. The export contains the final model and aggregate reports, not training splits or intermediate checkpoints.

In [ ]:
from zipfile import ZIP_DEFLATED
export = Path("/content/distilbert-results.zip")
lock = subprocess.check_output([sys.executable, "-m", "pip", "freeze"], text=True)
Path("reports/distilbert/requirements-colab-lock.txt").write_text(lock)
with ZipFile(export, "w", compression=ZIP_DEFLATED) as archive:
    for folder in [Path("models/distilbert/final"), Path("reports/distilbert")]:
        for item in sorted(folder.rglob("*")):
            if item.is_file():
                archive.write(item, item.as_posix())
    archive.write("models/distilbert/experiment.json")
files.download(str(export))